In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder\
    .appName("Discovery")\
    .master("spark://spark-master:7077")\
    .getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/25 20:44:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
print(spark.conf.get("spark.sql.extensions", ""))


io.delta.sql.DeltaSparkSessionExtension,org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions


In [4]:
df = spark.createDataFrame([(1, "Alice"), (2, "Bob")], ["id", "nome"])

df.write \
  .format("iceberg") \
  .mode("overwrite") \
  .saveAsTable("local.db_teste_iceberg")

In [5]:
print(spark.conf.get("spark.sql.extensions"))
print(spark.conf.get("spark.sql.catalog.local"))
print(spark.conf.get("spark.sql.catalog.local.type"))
print(spark.conf.get("spark.sql.catalog.local.warehouse"))


io.delta.sql.DeltaSparkSessionExtension,org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
org.apache.iceberg.spark.SparkCatalog
hadoop
hdfs://namenode:8020/user/hive/warehouse


In [6]:
df = spark.sql("SELECT * FROM local.db_teste_iceberg")
df.show()


+---+-----+
| id| nome|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+



In [7]:
spark.sql("DESCRIBE EXTENDED local.db_teste_iceberg").show(truncate=False)


+----------------------------+----------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                             |comment|
+----------------------------+----------------------------------------------------------------------------------------------------------------------+-------+
|id                          |bigint                                                                                                                |NULL   |
|nome                        |string                                                                                                                |NULL   |
|                            |                                                                                                                      |       |
|# Metadata Columns          |                      

In [8]:
# Conte os registros da tabela
spark.sql("SELECT COUNT(*) FROM local.db_teste_iceberg").show()

# Veja os dados
spark.sql("SELECT * FROM local.db_teste_iceberg").show()

# Veja histórico de snapshots
spark.sql("SELECT * FROM local.db_teste_iceberg.history").show(truncate=False)

# Veja arquivos que compõem a tabela
spark.sql("SELECT * FROM local.db_teste_iceberg.files").show(truncate=False)


+--------+
|count(1)|
+--------+
|       2|
+--------+

+---+-----+
| id| nome|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+

+-----------------------+-------------------+---------+-------------------+
|made_current_at        |snapshot_id        |parent_id|is_current_ancestor|
+-----------------------+-------------------+---------+-------------------+
|2025-09-25 19:39:58.983|2541109954990277374|NULL     |false              |
|2025-09-25 19:41:03.108|252962479798779823 |NULL     |false              |
|2025-09-25 20:44:30.881|2218702063369475257|NULL     |true               |
+-----------------------+-------------------+---------+-------------------+

+-------+---------------------------------------------------------------------------------------------------------------------------+-----------+-------+------------+------------------+------------------+----------------+-----------------+----------------+-------------------------------------------------------+--------------------------

In [9]:
# 1. Criar o database/schema no catálogo Delta
spark.sql("""
    CREATE DATABASE IF NOT EXISTS spark_catalog.delta_db
    LOCATION 'hdfs://namenode:8020/user/hive/warehouse/delta_db'
""")

# 2. Salvar DataFrame no HDFS como Delta
df.write.format("delta").mode("overwrite").save(
    "hdfs://namenode:8020/user/hive/warehouse/delta_db/tabela_teste_delta"
)

# 3. Registrar a tabela Delta no catálogo
spark.sql("""
    CREATE TABLE IF NOT EXISTS spark_catalog.delta_db.tabela_teste_delta
    USING delta
    LOCATION 'hdfs://namenode:8020/user/hive/warehouse/delta_db/tabela_teste_delta'
""")

# 4. Consultar a tabela via SQL
spark.sql("SELECT * FROM spark_catalog.delta_db.tabela_teste_delta").show()

# 5. (Opcional) Ler a tabela direto do caminho HDFS
spark.read.format("delta").load(
    "hdfs://namenode:8020/user/hive/warehouse/delta_db/tabela_teste_delta"
).show(truncate=False)


25/09/25 20:44:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+---+-----+
| id| nome|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+

+---+-----+
|id |nome |
+---+-----+
|1  |Alice|
|2  |Bob  |
+---+-----+

